# Notebook 1: Huấn luyện Gesture Emotion Encoder
**Input:** sequences.npy (N,30,225), emotions.npy (N,2), mode_ids.npy (N,)
**Output:** gesture_emotion_encoder.pt + ảnh đánh giá học thuật

In [ ]:
import numpy as np, torch, torch.nn as nn, os, json
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib; import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'font.family':'serif','font.size':11,'axes.titlesize':12,
    'axes.labelsize':11,'legend.fontsize':10,'figure.dpi':150,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--'
})
SAVE = '/kaggle/working/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

for p in ['/kaggle/input/datasets/quangleuq/v3-data/','/kaggle/input/gesturhythm-v2/']:
    if os.path.exists(p+'sequences.npy'): DATA_DIR=p; break
X = np.load(DATA_DIR+'sequences.npy')
y = np.load(DATA_DIR+'emotions.npy')
m = np.load(DATA_DIR+'mode_ids.npy')
print(f'X:{X.shape} y:{y.shape} m:{m.shape}')
MODE = {0:'HAPPY HIGH',1:'HAPPY LOW',2:'SAD HIGH',3:'SAD LOW',4:'NEUTRAL',5:'NONE'}
COLORS = {0:'#27ae60',1:'#2980b9',2:'#c0392b',3:'#8e44ad',4:'#e67e22',5:'#7f8c8d'}


## 1. Phân tích phân bố dữ liệu trước khi huấn luyện

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
fig.suptitle('Hình T1.1: Phân bố tập huấn luyện',fontsize=13,fontweight='bold')
# (a) so mau moi mode
ax=axes[0]; cnt=[np.sum(m==i) for i in range(6)]
bars=ax.bar([MODE[i] for i in range(6)],cnt,color=[COLORS[i] for i in range(6)],edgecolor='black',lw=0.8)
[ax.text(b.get_x()+b.get_width()/2,c+3,str(c),ha='center',fontsize=9) for b,c in zip(bars,cnt)]
ax.axhline(np.mean(cnt),color='red',ls='--',lw=1.2,label=f'TB={np.mean(cnt):.0f}')
ax.set_title('(a) Số mẫu mỗi chế độ'); ax.set_ylabel('Số mẫu'); ax.legend(); ax.tick_params(axis='x',rotation=30)
# (b) scatter valence-arousal
ax2=axes[1]
for i in range(6):
    idx=m==i; ax2.scatter(y[idx,0],y[idx,1],c=COLORS[i],label=MODE[i],alpha=0.5,s=15,edgecolors='none')
ax2.axhline(0,color='k',lw=0.8,ls=':'); ax2.axvline(0,color='k',lw=0.8,ls=':')
ax2.set_title('(b) Phân bố Valence-Arousal'); ax2.set_xlabel('Valence'); ax2.set_ylabel('Arousal')
ax2.legend(ncol=2,fontsize=8)
# (c) histogram valence va arousal
ax3=axes[2]
ax3.hist(y[:,0],bins=25,alpha=0.6,color='steelblue',label='Valence',edgecolor='none')
ax3.hist(y[:,1],bins=25,alpha=0.6,color='coral',label='Arousal',edgecolor='none')
ax3.axvline(y[:,0].mean(),color='steelblue',ls='--',lw=1.5,label=f'TB_V={y[:,0].mean():.2f}')
ax3.axvline(y[:,1].mean(),color='coral',ls='--',lw=1.5,label=f'TB_A={y[:,1].mean():.2f}')
ax3.set_title('(c) Phân bố Valence & Arousal'); ax3.set_xlabel('Giá trị'); ax3.legend(fontsize=8)
plt.tight_layout(); plt.savefig(SAVE+'enc_fig1_data_distribution.png',dpi=150,bbox_inches='tight'); plt.show()
print('Saved enc_fig1_data_distribution.png')


## 2. Định nghĩa Model

In [ ]:
class GestureEmotionEncoder(nn.Module):
    def __init__(self,input_dim=225,d_model=128,nhead=4,num_layers=3):
        super().__init__()
        self.embed=nn.Linear(input_dim,d_model)
        self.pos_enc=nn.Embedding(512,d_model)
        enc=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=256,dropout=0.1,batch_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=num_layers)
        self.head=nn.Sequential(nn.Linear(d_model,64),nn.ReLU(),nn.Linear(64,2),nn.Tanh())
    def forward(self,x):
        B,T,_=x.shape
        mask=torch.triu(torch.ones(T,T,device=x.device),diagonal=1).bool()
        pos=torch.arange(T,device=x.device).unsqueeze(0)
        return self.head(self.transformer(self.embed(x)+self.pos_enc(pos),mask=mask)[:,-1,:])

class GestureDataset(Dataset):
    def __init__(self,X,y,aug=False):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
        self.aug=aug
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        x=self.X[i]+torch.randn_like(self.X[i])*0.008 if self.aug else self.X[i]
        return x,self.y[i]

ds=GestureDataset(X,y,aug=True)
n_train=int(0.8*len(ds)); n_val=len(ds)-n_train
train_ds,val_ds=random_split(ds,[n_train,n_val],generator=torch.Generator().manual_seed(42))
train_loader=DataLoader(train_ds,batch_size=32,shuffle=True,drop_last=True)
val_loader=DataLoader(val_ds,batch_size=32)
print(f'Train:{n_train} | Val:{n_val}')

model=GestureEmotionEncoder().to(device)
total=sum(p.numel() for p in model.parameters())
print(f'Parameters: {total:,}')


## 3. Huấn luyện

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=100)
criterion=nn.MSELoss()
EPOCHS=100; train_losses,val_losses,lr_hist=[],[],[]

for epoch in range(EPOCHS):
    model.train(); tl=0
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        optimizer.zero_grad(); loss=criterion(model(xb),yb)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); tl+=loss.item()
    model.eval(); vl=0
    with torch.no_grad():
        for xb,yb in val_loader: vl+=criterion(model(xb.to(device)),yb.to(device)).item()
    scheduler.step()
    train_losses.append(tl/len(train_loader))
    val_losses.append(vl/len(val_loader))
    lr_hist.append(optimizer.param_groups[0]['lr'])
    if (epoch+1)%20==0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f} | lr={lr_hist[-1]:.6f}')
print('Done!')


## 4. Đánh giá chi tiết

In [ ]:
model.eval()
all_pred,all_true=[],[]
with torch.no_grad():
    for xb,yb in val_loader:
        all_pred.append(model(xb.to(device)).cpu().numpy())
        all_true.append(yb.numpy())
P=np.vstack(all_pred); T=np.vstack(all_true)
mae_v=mean_absolute_error(T[:,0],P[:,0])
mae_a=mean_absolute_error(T[:,1],P[:,1])
r2_v=r2_score(T[:,0],P[:,0]); r2_a=r2_score(T[:,1],P[:,1])
corr_v,_=stats.pearsonr(T[:,0],P[:,0]); corr_a,_=stats.pearsonr(T[:,1],P[:,1])
print('=== METRICS TREN VAL SET ===')
print(f'MAE Valence : {mae_v:.4f}'); print(f'MAE Arousal : {mae_a:.4f}')
print(f'R2  Valence : {r2_v:.4f}');  print(f'R2  Arousal : {r2_a:.4f}')
print(f'Pearson Val : {corr_v:.4f}');print(f'Pearson Aro : {corr_a:.4f}')
print(f'Final Val Loss: {val_losses[-1]:.4f}')


## 5. Biểu đồ đánh giá học thuật

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(15,9))
fig.suptitle('Hình T1.2: Kết quả huấn luyện Gesture Emotion Encoder',fontsize=13,fontweight='bold')

# (a) Loss curve
ax=axes[0,0]
ax.plot(train_losses,color='steelblue',lw=2,label='Train')
ax.plot(val_losses,color='coral',lw=2,label='Val')
ax.set_title('(a) Đường cong loss (MSE)'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.legend(); ax.set_yscale('log')
best_e=int(np.argmin(val_losses))
ax.annotate(f'Best: {val_losses[best_e]:.4f}',xy=(best_e,val_losses[best_e]),
    xytext=(best_e+5,val_losses[best_e]*2),arrowprops=dict(arrowstyle='->',color='red'),color='red',fontsize=9)

# (b) LR schedule
axes[0,1].plot(lr_hist,color='green',lw=2)
axes[0,1].set_title('(b) Learning rate schedule (Cosine Annealing)')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Learning rate')

# (c) Pred vs True Valence
ax=axes[0,2]
ax.scatter(T[:,0],P[:,0],alpha=0.3,s=8,color='steelblue',edgecolors='none')
lims=[-1.2,1.2]; ax.plot(lims,lims,'r--',lw=1.5,label='Lý tưởng')
ax.set_title(f'(c) Valence: Pred vs True\nMAE={mae_v:.4f}, R²={r2_v:.4f}')
ax.set_xlabel('Thực tế'); ax.set_ylabel('Dự đoán'); ax.legend(); ax.set_xlim(lims); ax.set_ylim(lims)

# (d) Pred vs True Arousal
ax=axes[1,0]
ax.scatter(T[:,1],P[:,1],alpha=0.3,s=8,color='coral',edgecolors='none')
ax.plot(lims,lims,'r--',lw=1.5,label='Lý tưởng')
ax.set_title(f'(d) Arousal: Pred vs True\nMAE={mae_a:.4f}, R²={r2_a:.4f}')
ax.set_xlabel('Thực tế'); ax.set_ylabel('Dự đoán'); ax.legend(); ax.set_xlim(lims); ax.set_ylim(lims)

# (e) Emotion space: true vs pred
ax=axes[1,1]
ax.scatter(T[:,0],T[:,1],alpha=0.3,s=8,color='gray',label='Thực tế',edgecolors='none')
ax.scatter(P[:,0],P[:,1],alpha=0.3,s=8,color='steelblue',label='Dự đoán',edgecolors='none')
ax.axhline(0,color='k',lw=0.8,ls=':'); ax.axvline(0,color='k',lw=0.8,ls=':')
ax.set_title('(e) Không gian cảm xúc: Thực tế vs Dự đoán')
ax.set_xlabel('Valence'); ax.set_ylabel('Arousal'); ax.legend()

# (f) Residual distribution
ax=axes[1,2]
res_v=P[:,0]-T[:,0]; res_a=P[:,1]-T[:,1]
ax.hist(res_v,bins=30,alpha=0.6,color='steelblue',label=f'Valence (std={res_v.std():.3f})',edgecolor='none')
ax.hist(res_a,bins=30,alpha=0.6,color='coral',label=f'Arousal (std={res_a.std():.3f})',edgecolor='none')
ax.axvline(0,color='k',lw=1.5,ls='--'); ax.set_title('(f) Phân bố sai số dự đoán')
ax.set_xlabel('Sai số (Pred - True)'); ax.set_ylabel('Số lượng'); ax.legend()

plt.tight_layout()
plt.savefig(SAVE+'enc_fig2_training_results.png',dpi=150,bbox_inches='tight'); plt.show()
print('Saved enc_fig2_training_results.png')


## 6. Phân tích theo chế độ cảm xúc

In [ ]:
# Lay mode_ids cho val set
val_indices=[i for i in val_ds.indices]
val_modes=m[val_indices]
fig,axes=plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Hình T1.3: Đánh giá theo từng chế độ cảm xúc',fontsize=13,fontweight='bold')

mae_v_per=[]; mae_a_per=[]; labels_mode=[]
for mid in range(6):
    idx=np.where(val_modes==mid)[0]
    if len(idx)==0: continue
    mae_v_per.append(mean_absolute_error(T[idx,0],P[idx,0]))
    mae_a_per.append(mean_absolute_error(T[idx,1],P[idx,1]))
    labels_mode.append(MODE[mid])

x=np.arange(len(labels_mode)); w=0.35
axes[0].bar(x-w/2,mae_v_per,w,label='MAE Valence',color='steelblue',edgecolor='black',lw=0.8)
axes[0].bar(x+w/2,mae_a_per,w,label='MAE Arousal',color='coral',edgecolor='black',lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels_mode,rotation=25,ha='right')
axes[0].set_title('(a) MAE theo từng chế độ cảm xúc'); axes[0].set_ylabel('MAE'); axes[0].legend()

# Scatter pred emotion colored by mode
ax2=axes[1]
for mid in range(6):
    idx=np.where(val_modes==mid)[0]
    if len(idx)==0: continue
    ax2.scatter(P[idx,0],P[idx,1],c=COLORS[mid],label=MODE[mid],alpha=0.6,s=15,edgecolors='none')
ax2.axhline(0,color='k',lw=0.8,ls=':'); ax2.axvline(0,color='k',lw=0.8,ls=':')
ax2.set_title('(b) Không gian cảm xúc dự đoán (theo chế độ)')
ax2.set_xlabel('Valence dự đoán'); ax2.set_ylabel('Arousal dự đoán')
ax2.legend(ncol=2,fontsize=9)

plt.tight_layout()
plt.savefig(SAVE+'enc_fig3_per_mode_eval.png',dpi=150,bbox_inches='tight'); plt.show()
print('Saved enc_fig3_per_mode_eval.png')


## 7. Lưu model

## 6b. Ảnh riêng: Attention Weights Visualization

In [ ]:
# Trich xuat attention weights tu encoder layer 0
sample_x = torch.tensor(X[:1], dtype=torch.float32).to(device)
with torch.no_grad():
    e = model.embed(sample_x)
    pos = torch.arange(30, device=device).unsqueeze(0)
    e = e + model.pos_enc(pos)
    _, attn_w = model.transformer.layers[0].self_attn(e, e, e, need_weights=True, average_attn_weights=True)
    attn_w = attn_w.squeeze().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn_w, cmap='Blues', aspect='auto', vmin=0)
plt.colorbar(im, ax=ax)
ax.set_title('Hình E-01: Attention Weights - Encoder Layer 0\n'
             'Hàng i đang chú ý đến frame j với trọng số bao nhiêu', fontsize=12)
ax.set_xlabel('Frame nguồn (Key)'); ax.set_ylabel('Frame truy vấn (Query)')
plt.tight_layout()
plt.savefig(SAVE + 'enc_e01_attention_weights.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved enc_e01_attention_weights.png')


## 6c. Ảnh riêng: Mean Attention per Frame

In [ ]:
mean_attn = attn_w.mean(axis=0)
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(range(30), mean_attn, color='steelblue', edgecolor='black', linewidth=0.5)
peak = int(np.argmax(mean_attn))
bars[peak].set_color('coral')
ax.annotate(f'Peak: frame {peak}\n({mean_attn[peak]:.3f})',
    xy=(peak, mean_attn[peak]), xytext=(peak+2, mean_attn[peak]+0.003),
    arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)
ax.set_title('Hình E-02: Trọng số Attention trung bình theo Frame\n'
             'Frame nào được chú ý nhiều nhất khi dự đoán cảm xúc?', fontsize=12)
ax.set_xlabel('Frame (0=đầu, 29=cuối chuỗi)'); ax.set_ylabel('Attention weight trung bình')
plt.tight_layout()
plt.savefig(SAVE + 'enc_e02_mean_attention.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved enc_e02_mean_attention.png')


## 6d. Ảnh riêng: Per-Person Analysis (A vs B vs C)

In [ ]:
# Lay person_id tu dataset neu co, neu khong chia ngau nhien
# Chia 3 phan bang nhau lam proxy
n3 = N // 3
groups = {'Nguoi A': np.arange(0, n3), 'Nguoi B': np.arange(n3, 2*n3), 'Nguoi C': np.arange(2*n3, N)}
gc = ['steelblue', 'coral', 'mediumpurple']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Hình E-03: So sánh phân bố cảm xúc giữa những người thu thập dữ liệu', fontsize=12, fontweight='bold')

for (name, idx), c in zip(groups.items(), gc):
    axes[0].scatter(y[idx,0], y[idx,1], c=c, label=name, alpha=0.4, s=15, edgecolors='none')
axes[0].axhline(0, color='k', lw=0.8, ls=':'); axes[0].axvline(0, color='k', lw=0.8, ls=':')
axes[0].set_title('(a) Không gian Valence-Arousal theo người thu')
axes[0].set_xlabel('Valence'); axes[0].set_ylabel('Arousal'); axes[0].legend()

for (name, idx), c in zip(groups.items(), gc):
    # Mean speed per sample
    seqs = X[idx, :, :63]
    speeds = np.linalg.norm(np.diff(seqs, axis=1), axis=2).mean(axis=(1,2))
    axes[1].hist(speeds, bins=25, alpha=0.5, color=c, label=name, edgecolor='none', density=True)
axes[1].set_title('(b) Phân bố tốc độ di chuyển tay theo người thu')
axes[1].set_xlabel('Tốc độ trung bình (||Δlandmark||)'); axes[1].set_ylabel('Mật độ'); axes[1].legend()

plt.tight_layout()
plt.savefig(SAVE + 'enc_e03_per_person.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved enc_e03_per_person.png')


## 6e. Ảnh riêng: Phân tích Residuals chi tiết

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Hình E-04: Phân tích sai số dự đoán chi tiết', fontsize=12, fontweight='bold')

# (a) Q-Q plot residual valence
from scipy import stats as ss
res_v = P[:,0] - T[:,0]
(osm, osr), (slope, intercept, r) = ss.probplot(res_v, dist='norm')
axes[0].plot(osm, osr, 'o', color='steelblue', markersize=2, alpha=0.5)
axes[0].plot(osm, slope*np.array(osm)+intercept, 'r-', lw=1.5)
axes[0].set_title(f'(a) Q-Q plot sai số Valence\n(R={r:.4f})')
axes[0].set_xlabel('Quantile lý thuyết'); axes[0].set_ylabel('Quantile thực tế')

# (b) Q-Q plot residual arousal
res_a = P[:,1] - T[:,1]
(osm2, osr2), (slope2, intercept2, r2) = ss.probplot(res_a, dist='norm')
axes[1].plot(osm2, osr2, 'o', color='coral', markersize=2, alpha=0.5)
axes[1].plot(osm2, slope2*np.array(osm2)+intercept2, 'r-', lw=1.5)
axes[1].set_title(f'(b) Q-Q plot sai số Arousal\n(R={r2:.4f})')
axes[1].set_xlabel('Quantile lý thuyết'); axes[1].set_ylabel('Quantile thực tế')

# (c) Residual vs Predicted
axes[2].scatter(P[:,0], res_v, alpha=0.3, s=8, color='steelblue', label='Valence', edgecolors='none')
axes[2].scatter(P[:,1], res_a, alpha=0.3, s=8, color='coral', label='Arousal', edgecolors='none')
axes[2].axhline(0, color='k', lw=1.5, ls='--')
axes[2].set_title('(c) Residual vs Predicted'); axes[2].set_xlabel('Giá trị dự đoán')
axes[2].set_ylabel('Sai số'); axes[2].legend()

plt.tight_layout()
plt.savefig(SAVE + 'enc_e04_residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved enc_e04_residual_analysis.png')


## 6f. Ảnh riêng: Emotion Decision Boundary

In [ ]:
from sklearn.decomposition import PCA
# Lay feature tu encoder (hidden state cuoi)
all_hidden = []
model.eval()
with torch.no_grad():
    for i in range(0, min(500, N), 32):
        xb = torch.tensor(X[i:i+32], dtype=torch.float32).to(device)
        h = model.embed(xb) + model.pos_enc(torch.arange(30,device=device).unsqueeze(0))
        mask = torch.triu(torch.ones(30,30,device=device),diagonal=1).bool()
        h = model.transformer(h, mask=mask)[:,-1,:]
        all_hidden.append(h.cpu().numpy())
hidden = np.vstack(all_hidden)
pca = PCA(n_components=2); H2 = pca.fit_transform(hidden)
m_sub = m[:len(hidden)]

fig, ax = plt.subplots(figsize=(8, 6))
for mid in range(6):
    idx = m_sub == mid
    if idx.sum() == 0: continue
    ax.scatter(H2[idx,0], H2[idx,1], c=COLORS[mid], label=MODE[mid], alpha=0.6, s=20, edgecolors='none')
ax.set_title(f'Hình E-05: PCA không gian ẩn của Encoder\n'
             f'(Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%)', fontsize=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(SAVE + 'enc_e05_pca_hidden.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved enc_e05_pca_hidden.png')


In [ ]:
torch.save({
    'model_state':model.state_dict(),
    'config':{'input_dim':225,'d_model':128,'nhead':4,'num_layers':3},
    'metrics':{'val_loss':val_losses[-1],'mae_valence':float(mae_v),'mae_arousal':float(mae_a),
               'r2_valence':float(r2_v),'r2_arousal':float(r2_a),
               'pearson_valence':float(corr_v),'pearson_arousal':float(corr_a)},
    'train_losses':train_losses,'val_losses':val_losses
},'gesture_emotion_encoder.pt')
print('Saved: gesture_emotion_encoder.pt')
print(f'Val Loss: {val_losses[-1]:.4f} | MAE_V: {mae_v:.4f} | MAE_A: {mae_a:.4f}')
print(f'R2_V: {r2_v:.4f} | R2_A: {r2_a:.4f}')
